# Carlisle 2005: six hours of the flood against a full shallow-water reference

This notebook runs and analyses the test case `carlisle` from the case registry
(`cases/registry/carlisle.jl`; described in `docs/TEST_CASES.md` §3 and
`docs/HYDRAULICS.md` §15.10–15.11). It is the third of three notebooks:
`01_planar` (synthetic slopes), `02_channel` (a straight channel) and this one,
a real river.

The case covers the first 6 hours of the January 2005 flood on the River Eden at
Carlisle, with inflows on the Eden, Caldew and Petteril and 10 mm/h of rain on
every cell. It runs the event on two meshes of about the same cell size:

- **A5 resolution 18**: pentagonal cells of about 493 m², the area of a 22 m square;
- **Cartesian 20 m**: square cells aligned with the LiDAR, which has been averaged to 20 m.

On each mesh it compares FloodA5's default **local-inertial** solver with the
**full shallow-water (SWE) reference** solver. It also runs the **sub-grid (SGS)**
solver on A5. Two questions run through the notebook:

1. How close is the local-inertial default to the reference on each mesh?
2. Why does the local-inertial solver store about 25% more water than the
   reference on A5, but about the same as the reference (1% less) on Cartesian?

**How to run it.** Use the kernel *Julia FloodA5 1.12* (setup:
`notebooks/README.md`) and run all cells.

- **First run.** The notebook builds the meshes (about 7 min) and makes the five
  simulations in `test/case_runs/carlisle/`. With one 4-thread run at a time the
  simulations take about 45–50 min, most of it the two SWE runs.
- **Later runs.** A run is reused when its output file records the same command
  line and a completed run, so the notebook then takes a minute or two.
- **Analysis only.** Set `RUN_CASES = false` below to analyse existing runs
  without starting any.

## Setup

The notebook loads three modules, all from the repository:

| Module | File | Purpose |
|---|---|---|
| `FA` | `analysis/FloodA5Analysis.jl` | loads runs and computes metrics |
| `FC` | `cases/FloodA5Cases.jl` | the case registry and runner |
| `FP` | `visualisation/FloodA5Plots.jl` | static CairoMakie figures |

None of them loads the model or Python. The simulations run as separate
`julia FloodModel.jl …` processes.

The repository root comes from the active project. The kernel starts Julia with
`--project=<repo>`, so the notebook works from any checkout.

In [ ]:
const REPO = dirname(Base.active_project())
include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));   const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));         const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl")); const FP = FloodA5Plots
using CairoMakie, Printf, Statistics, Logging
import ArchGDAL as AG
CairoMakie.activate!(type = "png")   # PNG: an SVG of 30 000 cell polygons would be huge
FP.activate!()                       # the FloodA5 figure style

RUN_CASES = true    # false: analyse existing runs only (an error if one is missing)
PARALLEL  = 1       # simulations at a time
THREADS   = 4       # threads per simulation

c = FC.case("carlisle")
CARL = joinpath(REPO, "test", "carlisle")       # the tracked inputs
println("repository: ", REPO, "\ncase directory: ", FC.case_dir(c))

## 1. The case in the registry

In the registry, a case is plain data:

- the preparation steps (here, mesh generation);
- the runs, each one `FloodModel.jl` argument list;
- the metrics computed from the runs;
- the physical checks the metrics must pass;
- expected values for regression (`cases/expected/carlisle.toml`).

`FC.show_case` prints the case's description and every command exactly as the
runner executes it, so the text and the commands cannot drift apart. Each
command is what you would type from the repository root. Paths are expanded
for this checkout.

In [ ]:
FC.show_case(c)

## 2. The inputs

The case uses four tracked files in `test/carlisle/`:

| File | Content |
|---|---|
| `Carlisle_domain.geojson` | the area of interest (AOI): a 4.75 km × 3.06 km rectangle in British National Grid, stored in WGS 84 |
| `Carlisle_LiDAR_5m_mean.tif` | 5 m LiDAR terrain, British National Grid with ODN heights |
| `carlisle_wgs84.bci` | the boundary conditions: where water enters (LISFLOOD-FP format) |
| `carlisle_wgs84.bdy` | the inflow hydrographs the `.bci` refers to |

FloodA5 works in longitude and latitude on the sphere. The DEM stays in its own
coordinate system: the model reprojects its sample points into it, not the raster.

### 2.1 Domain and DEM

In [ ]:
aoi = read(joinpath(CARL, "Carlisle_domain.geojson"), String)
println("AOI properties: ", match(r"(\x22MINX[^}]*?)\s*\}", aoi).captures[1])

AG.read(joinpath(CARL, "Carlisle_LiDAR_5m_mean.tif")) do ds
    gt = AG.getgeotransform(ds)
    band = AG.getband(ds, 1)
    z = Float64.(AG.read(band)); nd = AG.getnodatavalue(band)
    v = filter(x -> isfinite(x) && (nd === nothing || x != nd), z)
    @printf("DEM: %d × %d pixels of %.0f m; elevation %.1f to %.1f m (median %.1f m)\n",
            AG.width(ds), AG.height(ds), gt[2], minimum(v), maximum(v), median(v))
    println("CRS: ", match(r"^\w+\[\x22([^\x22]+)\x22", AG.getproj(ds)).captures[1])
end

### 2.2 Inflows and rain

The `.bci` file has one line per boundary condition:

- **`P` lines** are point inflows: longitude, latitude, the code `QVAR` (a
  discharge that varies in time) and the name of the hydrograph in the `.bdy`
  file. The model adds each hydrograph to the cell nearest its point. The three
  points are the River Eden (`main`), the Caldew (`cummersdale`) and the
  Petteril (`harraby`).
- **The `W` line** is a LISFLOOD-FP "west edge" boundary. FloodA5 domains can be
  any polygon, so it ignores `N`/`E`/`S`/`W` lines with a warning (`--bc-file`
  places boundaries by location instead).

FloodA5 does not need the `W` line: every edge of the domain is already an open
boundary, a free outflow at normal depth (`--open-boundary normal`, the default
since 0.3.0; HYDRAULICS §15.11).

The `.bdy` file holds one block per hydrograph: the name, the number of rows,
the time unit, then rows of *value, time* (in that order: discharge first).
The cell below reads it with the model's own reader,
`boundaryinputs/timeseries_io.jl`.

In [ ]:
println(read(joinpath(CARL, "carlisle_wgs84.bci"), String))
println(join(first(readlines(joinpath(CARL, "carlisle_wgs84.bdy")), 7), "\n"), "\n…")

module BDY   # the model's .bdy reader, on its own
    include(joinpath(Main.REPO, "boundaryinputs", "timeseries_io.jl"))
end
series = with_logger(ConsoleLogger(stderr, Logging.Warn)) do
    BDY.read_timeseries(BDY.LisfloodBDYReader(joinpath(CARL, "carlisle_wgs84.bdy")))
end
RIVER = Dict("main" => "Eden (main)", "cummersdale" => "Caldew (cummersdale)", "harraby" => "Petteril (harraby)")
T_END = 6 * 3600.0
inflow_6h = 0.0
for n in ("main", "cummersdale", "harraby")
    t, q = series[n]
    v6 = sum((q[k] + q[k+1]) / 2 * (t[k+1] - t[k]) for k in 1:length(t)-1 if t[k+1] <= T_END)
    global inflow_6h += v6
    @printf("%-22s Q(0) = %6.1f  Q(6 h) = %6.1f m³/s; peak %6.1f m³/s at %4.1f h; volume in 6 h %.3f Mm³\n",
            RIVER[n], q[1], q[findfirst(>=(T_END), t)], maximum(q), t[argmax(q)] / 3600, v6 / 1e6)
end

In [ ]:
let fig = Figure(size = (900, 380))
    ax1 = Axis(fig[1, 1]; xlabel = "Time (h)", ylabel = "Discharge (m³/s)", title = "The whole event in the .bdy file",
               subtitle = "shaded: the 6 h simulated here")
    ax2 = Axis(fig[1, 2]; xlabel = "Time (h)", title = "The first 6 h", subtitle = "the inflows of every run")
    vspan!(ax1, [0.0], [6.0]; color = (FP.MUTED, 0.2))
    for (i, n) in enumerate(("main", "cummersdale", "harraby"))
        t, q = series[n]
        lines!(ax1, t ./ 3600, q; color = FP.SERIES[i], linewidth = 2, label = RIVER[n])
        s = t .<= T_END
        lines!(ax2, t[s] ./ 3600, q[s]; color = FP.SERIES[i], linewidth = 2)
    end
    Legend(fig[2, 1:2], ax1; orientation = :horizontal)
    fig
end

The 6 h window catches the rising limb. The Eden more than doubles, from about
100 to 230 m³/s, while the Caldew and Petteril stay roughly steady. The flood
peak (about 1 270 m³/s on the Eden, at 55 h) is outside the window.

**Rain.** `--rainfall 10` adds a uniform 10 mm/h to every cell for the whole run.
This is a test load, not the observed rain. It wets the whole floodplain, so
the comparison covers thin rain-on-grid sheet flow as well as river flow. Thin
sheet flow is where the solvers differ most (HYDRAULICS §15.10). §3 adds up the
water budget once the mesh area is known.

## 3. The meshes

The case prepares three meshes in `test/case_runs/carlisle/`. A preparation
step is skipped when its output files exist.

### 3.1 A5 resolution 18 (pya5)

`FloodModel.jl --meshgen AOI --meshres 18 --dem DEM --meshout MESH --mesh-only`
builds the mesh in three stages. It runs with `--threads 1` because of a
PyCall/GDAL hazard.

1. **Julia samples the AOI.** It lays a dense grid of points over the AOI's
   bounding box, at 0.45 of a cell width, and keeps the points inside the
   polygon. This point-in-polygon test runs on the GPU if one is available,
   otherwise on CPU threads.
2. **Python finds the cells** (`mesh/a5_bridge.py`, run with FloodA5's Python:
   PyCall's Conda environment, where pya5 is installed). In order:
   - pya5's `polygon_to_cells` (pya5 1.x) gives every resolution-18 cell that
     overlaps the AOI;
   - the set grows by each cell's neighbours whose polygon intersects the AOI,
     until nothing changes;
   - two cells are neighbours if they share two vertices, which is exact, unlike
     pya5's `grid_disk`;
   - each cell's area comes from its vertices.

   The bridge writes a GeoParquet file and records the cell-ID scheme
   (`a5-v1`) in its metadata.
3. **Julia samples the DEM.** Each cell's elevation is the mean of the DEM,
   interpolated bilinearly at up to 256 quasi-random (Halton) points inside the
   polygon (`A5Grid.sample_dem_mean!`). Cells with no DEM under them get NaN and
   take no part in the flow.

The core of stage 2:

In [ ]:
# Print a few lines of a source file, from the first line that matches `from`
function excerpt(file, from, n)
    L = readlines(joinpath(REPO, file))
    i = findfirst(l -> occursin(from, l), L)
    i === nothing && return println("(", file, ": ", repr(from), " not found)")
    println(file, ", lines ", i, "–", i + n - 1, ":\n")
    foreach(println, L[i:i+n-1])
end
excerpt("mesh/a5_bridge.py", "# ── Pass 1: seed cell set", 16)
println()
excerpt("mesh/a5_bridge.py", "# ── Pass 2: grid_disk expansion", 26)

### 3.2 SGS tables on the same A5 mesh

The SGS solver needs more than one elevation per cell. The SGS mesh is the same
A5 mesh, generated with `--flow-model sgs`, so that `A5Grid.build_sgs_tables!`
also stores two kinds of sub-grid table from the DEM.

- **Per cell:** 512 Halton points give a 100-bin hypsometric curve, the stored
  volume and wetted area as functions of water level.
- **Per shared edge:** the DEM is sampled every 2.5 m along the edge, half the
  DEM pixel size. This gives the edge's sill elevation and its flow-area and
  wetted-perimeter curves.

The SGS solver measures depth from the cell's lowest DEM point and fills the
cell's low parts first, such as a channel narrower than the cell. Flow between
two cells is limited by the edge's own profile.

### 3.3 Cartesian 20 m

`cases/scripts/average_dem.jl` averages the 5 m LiDAR onto a 20 m grid
(`gdalwarp -tr 20 20 -r average -tap`: each 20 m cell is the mean of its 16
pixels, on a grid snapped to multiples of 20 m). `--meshgen-cartesian` then
turns each pixel into a cell whose elevation is the pixel value. This step needs
no Python.

The A5 cells are about 23% larger in area than the Cartesian cells, so the two
meshes have similar, but not equal, resolution.

The next cell runs the missing preparation steps. It prints "up to date" for
steps whose outputs exist. Then it prints the key lines of each step's log.

In [ ]:
FC.prepare(c)
println()
keylines = r"Sample points:|Points inside AOI|Python bridge:|Mesh generated in|outside DEM extent|mean sampling complete|SGS pre-processing:|edge sample spacing|SGS tables built in \d|Raster-aligned Cartesian grid:|cells of \d+ m"
for p in c.prep
    log = joinpath(FC.case_dir(c), "prep_$(p.name).log")
    isfile(log) || continue
    println("── ", p.name)
    for l in eachline(log)
        occursin(keylines, l) || continue
        l = replace(l, r"[A-Za-z]:[\\/][^\s:]*[\\/]" => "", r"^[┌└│\[ ]*(Info:|Warning:)\s*" => "")
        println("   ", strip(l))
    end
end

The figure shows the two meshes' elevations, read from their GeoParquet files.
The zoom covers the Caldew joining the Eden:

- the A5 cells follow no axis, and a channel crosses them at every angle;
- the Cartesian grid staircases every channel that is not aligned with x or y.

The narrow Caldew and Petteril are only one or two cells wide on both meshes.

In [ ]:
a5_mesh   = FA.load_mesh_parquet(joinpath(FC.case_dir(c), "carlisle_a5_res18.parquet"))
cart_mesh = FA.load_mesh_parquet(joinpath(FC.case_dir(c), "carlisle_cart20.parquet"))
for (name, m) in (("A5 res 18", a5_mesh), ("Cartesian 20 m", cart_mesh))
    @printf("%-15s %6d cells, mean area %5.1f m² (a %.1f m square), domain %.2f km², %d cells without elevation\n",
            name, length(m), mean(m.area), sqrt(mean(m.area)), sum(m.area) / 1e6, count(isnan, m.elevation))
end

# A map of a mesh alone: FP draws runs, so wrap the mesh in a one-frame Run
mesh_run(m, label) = FA.Run("", label, 2, [0.0], m, Dict{String,Any}(), "")
let fig = Figure(), rng = (8.0, 32.0), box = (-2.9480, -2.9380, 54.8985, 54.9045)
    for (j, r) in enumerate((mesh_run(a5_mesh, "A5 res 18"), mesh_run(cart_mesh, "Cartesian 20 m")))
        FP.fieldmap!(fig[1, j], r, 1; var = :elevation, colormap = :batlowW, colorrange = rng, sources = false,
                     basemap = :none, title = r.label, subtitle = "cell elevation, $(length(r.mesh)) cells",
                     width = 430, colorbar = j == 2)
        p = FP.fieldmap!(fig[2, j], r, 1; var = :elevation, colormap = :batlowW, colorrange = rng, sources = false,
                         basemap = :none, title = r.label * ": zoom", subtitle = "the Caldew joining the Eden",
                         width = 430, colorbar = false)
        limits!(p.axis, box...)
        p.axis.aspect = AxisAspect((box[2] - box[1]) * cosd(54.9) / (box[4] - box[3]))
    end
    resize_to_layout!(fig)
    fig
end

**The water budget.** The three hydrographs and the rain, over the A5 mesh's
area, add up to the volume every A5 run should receive in 6 h. §4 compares this
with each run's own count.

In [ ]:
rain_6h = sum(a5_mesh.area) * 0.010 * 6     # 10 mm/h for 6 h
@printf("inflows %.3f Mm³ + rain %.3f Mm³ = %.3f Mm³\n", inflow_6h / 1e6, rain_6h / 1e6, (inflow_6h + rain_6h) / 1e6)

## 4. The runs and the checks

The case makes five runs:

| Run | Mesh | Solver | Role |
|---|---|---|---|
| `a5_swe` | A5 | `--flow-model swe` | the reference on A5 (see below) |
| `a5_li` | A5 | `--flow-model standard` | the 0.3.0 local-inertial default; compared with `a5_swe` |
| `cart_swe` | Cartesian | `--flow-model swe` | the reference on Cartesian |
| `cart_li` | Cartesian | `--flow-model standard` | local-inertial; compared with `cart_swe` |
| `a5_sgs` | A5 + SGS tables | `--flow-model sgs` | the sub-grid solver |

**The SWE reference** solves the full 2-D shallow-water equations, advection
included. It is a second-order HLLC finite-volume scheme with hydrostatic
reconstruction (HYDRAULICS §15.10). It runs on the same mesh, bed, sources and
boundaries as the local-inertial solver, so it isolates what the local-inertial
simplification does on that mesh. It converges in its own numerical parameters:
halving its Courant number or adding eddy viscosity changes its 6 h storage by
less than 0.1%.

**The SGS run** has no reference with a sub-grid bed, so it is compared with the
others by storage only.

**Common options:**

| Option | Meaning |
|---|---|
| `--inflow-bci` | the three hydrographs (§2.2) |
| `--rainfall 10` | 10 mm/h on every cell |
| `--sim-duration 21600` | 6 h |
| `--dt-max 10` | the time step is adaptive and at most 10 s |
| `--output-interval 900` | a frame every 15 min |

The runner appends `--output test/case_runs/carlisle/runs/<run>.h5` and writes a
log next to it. Every output file records its command (`FO.run_info(path)["argv"]`).
`julia --project=. FloodModel.jl --rerun <run>.h5` reproduces a run bit for bit.

The commands:

In [ ]:
for j in c.jobs
    println(j.name, "   (", join(("$k = $v" for (k, v) in j.tags), ", "), ")")
    println("  julia --project=. --threads ", j.threads, " FloodModel.jl ", join(FC.job_argv(c, j), " "), "\n")
end

The next cell runs the case: it prepares the meshes, makes the runs that are
missing or out of date, analyses them, and checks them. On the first run this
takes most of an hour, and the runs' progress is in their logs
(`test/case_runs/carlisle/runs/*.log`). With `RUN_CASES = false` it only
analyses and checks the existing runs.

The checks, with the recorded values from HYDRAULICS §15.10–15.11:

- **Mass conservation:** the mass-balance error is below 10⁻⁴ of the volume added.
- **Reference storage:** the SWE storage at 6 h is within 1% of the recorded
  values (A5 2.690, Cartesian 2.331 Mm³).
- **Local-inertial storage:** the ratio to SWE is within ±0.05 of the recorded
  values (A5 1.25, Cartesian 0.99).
- **A5 vs Cartesian:** the A5 ratio exceeds the Cartesian one by more than 0.15.
- **Extent and depth:** CSI ≥ 0.55 and rms Δh ≤ 0.35 m against SWE.
- **Arrival:** the time at which 50 000 m³ has left the domain, within 2% of
  the recorded SWE values (6 688 and 5 736 s).
- **SGS storage:** within 2% of the recorded 4.30 Mm³.

Finally, every metric is compared with `cases/expected/carlisle.toml`, a
regression test. A5 results are sensitive to round-off in the bed. A mesh
regenerated on another machine can move them beyond those tight tolerances
while the physical checks still pass; if so, re-record the expected values
(`cases/run.jl baseline carlisle`).

In [ ]:
t0 = time()
ok = RUN_CASES ? FC.run_case(c; parallel = PARALLEL, threads = THREADS) : FC.check(c)
@printf("\n%s (%.1f min)\n", ok ? "All checks pass." : "SOME CHECKS FAIL: see above.", (time() - t0) / 60)

The case's metrics for each run. The storage, volumes added and removed, and
step counts come from each output file's end-of-run record. The comparisons
with SWE are at 6 h on the same mesh. Δh and CSI use the conventions of
`FA.compare`:

- **rms Δh:** over cells wetter than 1 cm in either run;
- **CSI** (critical success index): wet means deeper than 0.1 m, and
  CSI = hits / (hits + misses + false alarms);
- **bias:** the run's wet cells divided by the reference's wet cells.

In [ ]:
m = FC.analyse(c; io = devnull)
R = Dict(j.name => FA.load_run(FC.job_output(c, j); label = j.name) for j in c.jobs)
RECORDED = Dict("a5_swe" => "2.690", "cart_swe" => "2.331", "a5_sgs" => "4.30")   # HYDRAULICS §15.10–15.11 (Mm³)
@printf("%-9s %8s %8s %6s %7s %5s %5s %8s %8s %8s %9s %8s %7s\n", "run", "storage", "record", "÷ SWE",
        "rms Δh", "CSI", "bias", "arrival", "added", "removed", "mb error", "steps", "wall")
@printf("%-9s %8s %8s %6s %7s %5s %5s %8s %8s %8s %9s %8s %7s\n", "", "Mm³", "Mm³", "", "m", "", "", "s",
        "Mm³", "Mm³", "rel", "", "min")
for n in ("a5_swe", "a5_li", "cart_swe", "cart_li", "a5_sgs")
    x = m[n]; s = R[n].info["summary"]
    g(k, f) = haskey(x, k) ? Printf.format(Printf.Format(f), x[k]) : "–"   # @sprintf needs a literal format
    @printf("%-9s %8.3f %8s %6s %7s %5s %5s %8.0f %8.3f %8.3f %9.1e %8d %7.1f\n", n, x["storage_m3"] / 1e6,
            get(RECORDED, n, "–"), g("storage_ratio", "%.3f"), g("rms_dh", "%.3f"), g("csi", "%.2f"),
            g("bias", "%.2f"), x["arrival_s"], s["vol_added_m3"] / 1e6, s["vol_removed_m3"] / 1e6,
            x["mass_balance_rel"], s["n_steps"], s["wall_simulation_s"] / 60)
end

How to read the table:

- **Volume added.** Every A5 run receives the volume of the budget in §3. The
  Cartesian runs receive slightly less, because their domain (the 20 m raster's
  extent) is a little smaller.
- **Mass balance.** The `mb error` column, |added − stored − removed| ÷ added,
  is at round-off level in every run. The first check uses the log's step lines
  and is looser.
- **Storage is outflow.** Since stored = added − removed, a run that stores more
  has let less water out. The A5 local-inertial run stores 0.68 Mm³ more than
  the A5 reference because it has let about 0.68 Mm³ less out of the domain, and
  its arrival time is about 50 min later.
- **Cost.** Each SWE run takes 7–10 times the steps of the local-inertial run
  on the same mesh (the `wall` column is the simulation time of the run as it was
  made, on a shared machine). Its stable step is bounded by the wave speed |u| + √(gh),
  not by √(gh) alone.

## 5. Results

### 5.1 Storage through time

The plot shows the water stored in the domain at each 15-minute frame. Colour
marks the mesh (A5 blue, Cartesian orange) and line style the solver
(local-inertial solid, SWE dashed, SGS dotted). The SGS storage is the sum of
the cell volumes (`FA.storage`).

In [ ]:
LABEL = Dict("a5_swe" => "A5 SWE", "a5_li" => "A5 local-inertial", "cart_swe" => "Cartesian SWE",
             "cart_li" => "Cartesian local-inertial", "a5_sgs" => "A5 SGS")
for (n, r) in R   # reload with display labels (titles and legends use them)
    R[n] = FA.load_run(r.path; label = LABEL[n])
end
a5_swe, a5_li, cart_swe, cart_li, a5_sgs = (R[n] for n in ("a5_swe", "a5_li", "cart_swe", "cart_li", "a5_sgs"))
RUNS = [a5_li, a5_swe, a5_sgs, cart_li, cart_swe]

display(FP.storage_plot(RUNS))
println("storage (Mm³)             1 h     2 h     4 h     6 h")
for r in RUNS
    @printf("%-24s", r.label)
    for t in (1, 2, 4, 6) .* 3600.0
        @printf("%8.3f", FA.storage(r, FA.frame_at(r, t)) / 1e6)
    end
    println()
end

**Up to about 1.5 h** all five runs store the same volume, because almost
nothing has left the domain yet and they receive the same inputs.

**From about 1.5 h** they separate as water starts to leave, mainly down the
Eden at the western edge:

- On Cartesian, the local-inertial and SWE curves stay together all the way to 6 h.
- On A5, the local-inertial run stores more from 2 h onwards, and 1.25× the
  reference by 6 h.
- The A5 reference itself stores 15% more than the Cartesian reference
  (2.69 against 2.33 Mm³). HYDRAULICS §15.10 places A5 res 18 between the
  Cartesian 20 m and 40 m references in effective resolution. The cause is
  channels narrower than a cell, since the Caldew and Petteril are one or two
  cells wide.
- SGS stores the most, 4.30 Mm³ (§5.6).

### 5.2 Depth maps: local-inertial against SWE

The maps show depth at 2, 4 and 6 h. Cells shallower than 0.1 m show the
terrain instead: 0.1 m is the threshold the extent comparison uses, and it hides
the thin film the rain leaves everywhere. The stars mark the three inflow points.

In [ ]:
FP.fieldmaps([a5_li, a5_swe], [2, 4, 6] .* 3600; dry_threshold = 0.1, colorrange = (0, 5), panel = 330)

In [ ]:
FP.fieldmaps([cart_li, cart_swe], [2, 4, 6] .* 3600; dry_threshold = 0.1, colorrange = (0, 5), panel = 330)

On both meshes the rivers fill and the Eden's floodplain floods in the same
places in both solvers. The differences are in the detail:

- On A5, the local-inertial Eden is visibly deeper downstream (west) and spreads
  wider along its banks.
- On Cartesian, the two rows are hard to tell apart.
- In both SWE rows, many isolated cells deeper than 0.1 m are scattered across
  the floodplain. These are small hollows that the reference fills with rain.

### 5.3 Differences at 6 h

The two left-hand maps show local-inertial minus SWE depth, on cells wetter than
1 cm in either run, so their rms is the rms Δh of the table. Blue means the
local-inertial run is deeper. The extent maps classify each cell at the 0.1 m
threshold as wet in both (a hit), wet in the reference only (a miss), or wet in
the local-inertial run only (a false alarm).

In [ ]:
let fig = Figure()
    FP.difference_map!(fig[1, 1], a5_swe, a5_li; t = T_END, colorrange = (-1.5, 1.5), width = 460,
                       title = "A5: local-inertial − SWE")
    FP.difference_map!(fig[1, 2], cart_swe, cart_li; t = T_END, colorrange = (-1.5, 1.5), width = 460,
                       title = "Cartesian: local-inertial − SWE")
    resize_to_layout!(fig)
    fig
end

In [ ]:
let fig = Figure()
    FP.extent_map!(fig[1, 1], a5_swe, a5_li; t = T_END, width = 430)
    FP.extent_map!(fig[2, 1], cart_swe, cart_li; t = T_END, width = 430)
    resize_to_layout!(fig)
    fig
end

The next cell asks where the extra water is. It splits the 6 h storage
difference Σ (h − h_SWE)·A by the reference's depth in each cell:

In [ ]:
function storage_by_class(ref, run; t = T_END)
    kr, kk = FA.frame_at(ref, t), FA.frame_at(run, t)
    href, h, A = FA.depth(ref, kr), FA.depth(run, kk), ref.mesh.area
    @printf("%s − %s at %s: total %+.3f Mm³\n", run.label, ref.label, FP.hhmm(run.t[kk]),
            sum((h[i] - href[i]) * A[i] for i in eachindex(h) if isfinite(h[i]) && isfinite(href[i])) / 1e6)
    for (name, lo, hi) in (("dry or < 0.1 m in SWE", -Inf, 0.1), ("0.1–1 m", 0.1, 1.0),
                           ("1–3 m", 1.0, 3.0), ("> 3 m (river channels)", 3.0, Inf))
        sel = [i for i in eachindex(h) if isfinite(h[i]) && isfinite(href[i]) && lo < href[i] <= hi]
        @printf("   %-24s %6d cells   Δ storage %+.3f Mm³   mean Δh %+.3f m\n", name, length(sel),
                sum((h[i] - href[i]) * A[i] for i in sel; init = 0.0) / 1e6, isempty(sel) ? NaN : mean(h[sel] .- href[sel]))
    end
end
storage_by_class(a5_swe, a5_li)
storage_by_class(cart_swe, cart_li)

**On A5** the local-inertial run is deeper almost everywhere, and the excess of
0.68 Mm³ is spread over every depth class:

- **The largest share, about a third, is a thin film.** It lies on the
  24 000 cells that are dry or shallower than 0.1 m in the reference: about
  2 cm more water per cell, on average.
- **The shallow floodplain (0.1–1 m) holds about 0.2 Mm³.**
- **The deep river channels hold about 0.15 Mm³.** There the local-inertial
  water is 0.4 m deeper on average, and up to 1.5 m deeper along the lower Eden.
- **The extent comparison agrees.** The local-inertial run floods more of the
  floodplain beside the Eden (false alarms, bias 1.06).

**On Cartesian** the differences are small and of both signs. The deep channels
are about 0.1 m shallower in the local-inertial run, and the floodplain is about
the same. Most of the extent disagreement is scattered single cells, the
rain-filled hollows of the reference (misses, bias 0.85), not the river.

### 5.4 Time series at a river cell and a floodplain point

Two points, marked on the 6 h map:

- **River Eden:** the deepest cell of the A5 reference at 6 h.
- **Floodplain:** a point south of the Eden.

Each run uses its own cell nearest each point. The SGS run is left out, because
its depth is measured from each cell's lowest point and so is not comparable
with the others.

In [ ]:
h_end = FA.depth(a5_swe)
i_deep = argmax(replace(h_end, NaN => -Inf))
POINTS = [(lon = a5_swe.mesh.lon[i_deep], lat = a5_swe.mesh.lat[i_deep], label = "River Eden (deepest A5 SWE cell)"),
          (lon = -2.925, lat = 54.897, label = "Floodplain")]
foreach(p -> @printf("%-34s lon %.5f  lat %.5f\n", p.label, p.lon, p.lat), POINTS)
let fig = Figure()
    p = FP.fieldmap!(fig[1, 1], a5_swe, nothing; t = T_END, dry_threshold = 0.1, colorrange = (0, 5), width = 560)
    FP.mark_points!(p, POINTS; labels = [q.label for q in POINTS])
    resize_to_layout!(fig)
    fig
end

In [ ]:
display(FP.timeseries([a5_li, a5_swe, cart_li, cart_swe], POINTS))
println(rpad("depth at 6 h (m)", 26), join((rpad(p.label, 34) for p in POINTS)))
for r in (a5_li, a5_swe, cart_li, cart_swe)
    @printf("%-26s", r.label)
    for p in POINTS
        @printf("%-34.2f", FA.depth(r)[FA.source_cell(r.mesh, p.lon, p.lat)])
    end
    println()
end

**In the Eden channel** the two Cartesian runs and the A5 reference rise
together. The A5 local-inertial run rises later but ends deeper, by most of a
metre at 6 h: this is the backed-up lower Eden of §5.3. The A5 cell and the
nearest Cartesian cell are not the same piece of channel, so the two meshes'
depths differ even with the same solver.

**On the floodplain** water arrives in the local-inertial runs about an hour
earlier than in the reference runs, on both meshes, and is deeper at 6 h. Here
local-inertial differs from SWE on both meshes alike.

### 5.5 Why the A5 local-inertial run stores more

- **It is not the time step or the damping.** Halving the Courant number
  changes the A5 local-inertial storage by 0.2% (HYDRAULICS §15.11). Any added
  momentum damping only increases storage (§15.9–15.10).
- **The excess is water held back.** It comes from 0.68 Mm³ less outflow. It is
  not near the domain edges: in §15.10, 97% of it is more than 20 cells from the
  boundary. It is spread over every depth class (§5.3 above): the largest part
  is a thin film on cells the reference has nearly dry, and the deepest part is
  along the lower Eden.
- **The bed representation.** Both solvers see each cell as flat at its mean
  elevation, but they treat the step between two cells differently:
  - The local-inertial solver lets water cross a face only above the higher of
    the two beds, max(z_i, z_j). The cells form a staircase.
  - The second-order SWE reference rebuilds a sloping bed at each face.

  A thin film crossing a sloping face bed drains, but a step bed holds it in
  shallow hollows. A first-order SWE run, which also uses a step bed, shows the
  same pattern of excess, more strongly (§15.10).
- **Why A5 more than Cartesian is not settled.** Two findings point at narrow
  channels:
  - The A5 reference behaves like a coarser grid than the Cartesian one
    (§5.1; HYDRAULICS §15.10).
  - The straight-channel series (`02_channel.ipynb`, HYDRAULICS §15.12) found
    that for channels 0.5–4 cells wide, A5 has no advantage over Cartesian, and
    that A5 has preferred directions: at that series' test site it did worst for
    channels running 0–15° from east. The Eden's general course here is
    east–west, but A5's face directions differ between the dodecahedron faces,
    so it is untested whether that pattern applies at Carlisle.

**In short.** On Cartesian the local-inertial default stays within 1% of the
reference. On A5 at this resolution, HYDRAULICS §15.10 attributes most of the
25% excess to the step-bed representation of the terrain, not to the
local-inertial dynamics. Why that representation costs more on A5 than on
Cartesian is still open. HYDRAULICS §15.10 recommends judging the polygon
consistency of the hydraulics on synthetic problems, and of the topography on
resolution sequences and channel tests.

### 5.6 The SGS run

SGS stores 4.30 Mm³ at 6 h, more than either A5 run. The next cell checks
whether that water is held in the cells' low parts, below the cell-mean bed that
the other solvers use, or is higher water.

The map shows the SGS water surface minus the A5 local-inertial one, on the cells
deeper than 0.1 m in both runs; the two runs are on the same cells. It is
limited to those cells because an SGS depth is measured from the cell's lowest
point: nearly every SGS cell holds a little water there, and its water surface
then lies below the cell-mean bed.

In [ ]:
dwse = let hs = FA.depth(a5_sgs), hl = FA.depth(a5_li), ws = FA.wse(a5_sgs), wl = FA.wse(a5_li),
    V = FA.volume(a5_sgs), zbar = a5_sgs.mesh.elevation
    both = [i for i in eachindex(hs) if hs[i] > 0.1 && hl[i] > 0.1 && isfinite(ws[i]) && isfinite(wl[i])]
    below = [i for i in eachindex(ws) if isfinite(ws[i]) && isfinite(zbar[i]) && ws[i] <= zbar[i] && isfinite(V[i])]
    sr = R["a5_sgs"].info["summary"]; lr = R["a5_li"].info["summary"]
    @printf("stored %.3f Mm³ (A5 local-inertial %.3f); removed %.3f Mm³ (A5 local-inertial %.3f, A5 SWE %.3f)\n",
            FA.storage(a5_sgs) / 1e6, FA.storage(a5_li) / 1e6, sr["vol_removed_m3"] / 1e6, lr["vol_removed_m3"] / 1e6,
            R["a5_swe"].info["summary"]["vol_removed_m3"] / 1e6)
    @printf("water held below the cell-mean bed: %.3f Mm³ in %d cells\n", sum(V[below]) / 1e6, length(below))
    @printf("water surface, SGS − local-inertial, over %d cells wet in both: median %+.2f m, mean %+.2f m\n",
            length(both), median(ws[both] .- wl[both]), mean(ws[both] .- wl[both]))
    d = fill(NaN, length(ws)); d[both] .= ws[both] .- wl[both]
    d
end
# FP maps draw a Run: wrap the difference as the "elevation" of a one-frame mesh run
let g = a5_sgs.mesh, fig = Figure()
    dm = FA.MeshGeom(g.ids, g.lon, g.lat, dwse, g.area, g.polygons, g.neighbours, g.static, g.id_scheme)
    FP.fieldmap!(fig[1, 1], mesh_run(dm, "A5: SGS − local-inertial water surface"), 1; var = :elevation,
                 colormap = FP.DIVERGING, colorrange = (-1, 1), basemap = :grey, sources = false, width = 560,
                 subtitle = "t = 06:00 · cells deeper than 0.1 m in both", colorbar_label = "Δ water surface (m)")
    resize_to_layout!(fig)
    fig
end

Very little of the SGS excess is water hidden below the cell-mean bed. Most of
it is higher water: the SGS water surface is on average about 0.15 m above the
local-inertial one, and the SGS run lets only 1.6 Mm³ out against 2.5 (A5
local-inertial) and 3.2 Mm³ (A5 SWE). Its first 50 000 m³ of outflow arrives
between the two.

The map shows where the SGS water is higher and where it is lower:

- **Higher** along the Eden and its floodplain, by up to 1 m and more upstream
  (east), so the Eden drains more slowly with SGS.
- **Lower** along the Caldew and the Petteril. These narrow channels, one or two
  cells wide, are what the sub-grid tables should represent better, because
  water can fill the channel inside a cell.

A plausible cause of the slower Eden, not tested here, is that flow between SGS
cells passes only through each edge's own sub-grid profile above its sill.
Whether the SGS run is closer to the real event cannot be decided here: no
reference exists yet with a sub-grid bed (HYDRAULICS §15.11). An observed flood outline
for 2005 (TEST_CASES §3, "Validation data") is the planned test.

## 6. Summary and pointers

**Summary** (6 h, against the full shallow-water reference on the same mesh):

| | Cartesian 20 m | A5 res 18 |
|---|---|---|
| local-inertial storage | 0.99× SWE | 1.25× SWE |
| rms Δh | 0.13 m | 0.29 m |
| CSI | 0.69 | 0.62 |

- The local-inertial default matches the reference closely on Cartesian.
- On A5 it stores 25% more because it lets less water out. The excess is spread
  over every depth class, from a thin film on nearly dry cells to a backed-up
  lower Eden. HYDRAULICS §15.10 attributes it mainly to the step-bed
  representation, not to the local-inertial dynamics. Why the step bed costs
  more on A5 is open.
- The two references themselves differ by 15% between the meshes, a resolution
  effect of channels narrower than a cell.
- SGS stores the most. The cause is higher water and slower drainage, and there
  is no sub-grid reference to judge it against.
- Mass is conserved to round-off in every run.

**Replaying a run** in the interactive GLMakie viewer, with a frame slider and
the live viewer's controls. It needs the full stack (`--project=.`, without
`FLOODA5_NO_VIS`):

```bash
julia --project=. --threads auto visualisation/replay.jl test/case_runs/carlisle/runs/a5_li.h5
julia --project=. --threads auto visualisation/replay.jl test/case_runs/carlisle/runs/a5_swe.h5 --var velocity --from 10800
```

**From the shell:**

```bash
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl show carlisle            # description and commands
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl run carlisle -P 1 -T 4   # prepare, run, check
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl check carlisle           # check existing runs
julia --project=. FloodModel.jl --rerun test/case_runs/carlisle/runs/a5_li.h5   # reproduce one run
```

**Further reading:**

- `docs/TEST_CASES.md` §3: the Carlisle case.
- `docs/HYDRAULICS.md`:
  - §15.10: the SWE reference and the bed-representation analysis;
  - §15.11: the 0.3.0 defaults;
  - §15.12: the channel series.
- `docs/DATA_FORMATS.md`: the output file (§2) and the `.bci`/`.bdy` formats (§3).
- `docs/USER_GUIDE.md` §6.5: the figure functions used here.